# Notebook 02 : Annotation Preparation
Converts documents.json → spaCy DocBin (train/dev/test 80/10/10)

In [34]:
import json, random, os
import spacy
from spacy.tokens import DocBin

with open('../data/synthetic/documents.json') as f:
    documents = json.load(f)

# Split RESPECTING name pools — not random shuffle
train_pool = [d for d in documents if d['split'] == 'train']
test_pool  = [d for d in documents if d['split'] == 'test']

# Shuffle within each pool
random.seed(42)
random.shuffle(train_pool)
random.shuffle(test_pool)

# 80/20 split of train_pool → actual train + dev
n = len(train_pool)
train_docs = train_pool[:int(n * 0.85)]   # 5,100 docs
dev_docs   = train_pool[int(n * 0.85):]   # 900 docs
test_docs  = test_pool                     # 1,000 docs — fully unseen names

print(f"Train: {len(train_docs)} | Dev: {len(dev_docs)} | Test: {len(test_docs)}")
print(f"Test uses completely different name pool ✅")

Train: 108 | Dev: 20 | Test: 22
Test uses completely different name pool ✅


In [35]:
nlp = spacy.blank('en')

def remove_overlaps(entities):
    """
    Sort spans by start, then remove any span that overlaps with a previous one.
    Keep the longer span when two overlap.
    """
    # Sort by start position, then by length descending (longer span first)
    sorted_ents = sorted(entities, key=lambda x: (x[0], -(x[1]-x[0])))
    kept = []
    for ent in sorted_ents:
        # Check if this span overlaps with any already kept span
        overlap = any(
            ent[0] < k[1] and k[0] < ent[1]   # overlap condition
            for k in kept
        )
        if not overlap:
            kept.append(ent)
    return kept


def to_docbin(data, name):
    db      = DocBin()
    skipped = 0
    overlap_removed = 0

    for item in data:
        doc  = nlp.make_doc(item['text']) #  To tokenise the text
        
        # Remove overlapping spans BEFORE converting to spaCy spans
        clean_entities = remove_overlaps(item['entities'])
        overlap_removed += len(item['entities']) - len(clean_entities)

        ents = []
        for start, end, label in clean_entities:
            span = doc.char_span(start, end, label=label, alignment_mode='contract')
            if span is not None:
                ents.append(span)
            else:
                skipped += 1

        doc.ents = ents
        db.add(doc)

    print(f"{name}: {len(data)} docs | overlaps removed: {overlap_removed} | spans skipped: {skipped}")
    return db

os.makedirs('../data/annotated', exist_ok=True)
to_docbin(train_docs,'train').to_disk('../data/annotated/train.spacy')
to_docbin(dev_docs,  'dev'  ).to_disk('../data/annotated/dev.spacy')
to_docbin(test_docs, 'test' ).to_disk('../data/annotated/test.spacy')
print('Done')

train: 108 docs | overlaps removed: 75 | spans skipped: 4
dev: 20 docs | overlaps removed: 14 | spans skipped: 1
test: 22 docs | overlaps removed: 16 | spans skipped: 0
Done


In [36]:
# Add this verification cell RIGHT BEFORE the to_docbin block
print("Split sizes:")
print(f"  train: {len(train_docs)}")
print(f"  dev:   {len(dev_docs)}")
print(f"  test:  {len(test_docs)}")

# Verify name separation is working
train_names_used = set()
test_names_used  = set()

for d in train_docs:
    for s, e, lbl in d['entities']:
        if lbl == 'PERSON':
            train_names_used.add(d['text'][s:e])

for d in test_docs:
    for s, e, lbl in d['entities']:
        if lbl == 'PERSON':
            test_names_used.add(d['text'][s:e])

overlap = train_names_used & test_names_used
print(f"\nUnique PERSON names in train: {len(train_names_used)}")
print(f"Unique PERSON names in test:  {len(test_names_used)}")
print(f"Names appearing in BOTH:      {len(overlap)}")

if len(overlap) == 0:
    print("✅ Perfect — zero name overlap between train and test")
elif len(overlap) < 20:
    print(f"⚠️  Small overlap — {len(overlap)} shared names, acceptable")
else:
    print(f"❌ Too much overlap — {len(overlap)} shared names, check split logic")

Split sizes:
  train: 108
  dev:   20
  test:  22

Unique PERSON names in train: 155
Unique PERSON names in test:  31
Names appearing in BOTH:      0
✅ Perfect — zero name overlap between train and test


Errors: 
Error 1 : 
name  = "Sarah Kowalski"
email = "sarah.kowalski@gmail.com"

text = "...sarah.kowalski@gmail.com... Sarah Kowalski..."

collect() finds:
  (5,  19,  "PERSON")   ← "Sarah Kowalski" inside the email address ← WRONG
  (5,  29,  "EMAIL")    ← "sarah.kowalski@gmail.com"
  (32, 46,  "PERSON")   ← "Sarah Kowalski" standalone ← correct

Spans (5,19) and (5,29) overlap at token 100 → ERROR

Fix : Two Parts
Part 1: Fix collect() to skip matches inside longer strings
Part 2: Fix to_docbin() in Notebook 02 to filter overlaps before setting entities

In [37]:
# Run this in Notebook 02 — creates a real test set
import spacy
from spacy.tokens import DocBin

nlp = spacy.blank('en')

# Manually written docs — completely different style from templates
REAL_TEST_DOCS = [
    {
        "text": "hey spoke to david mcallister his number is 416-334-9821 "
                "email d.mcallister@hotmail.com account 00341-006-4421987 "
                "at BMO owes $3,200.00",
        "entities": [
            (14, 30, "PERSON"),
            (45, 57, "PHONE"),
            (64, 91, "EMAIL"),
            (100, 118, "ACCOUNT_NO"),
            (125, 128, "ORG"),
            (135, 144, "AMOUNT"),
        ]
    },
    {
        "text": "fatima al-hassan called re her td bank account. "
                "reach her at 514-882-3301 or fatima.h@yahoo.ca. "
                "address 221 rue sainte-catherine montreal qc h2x 1l4. "
                "disputed $890.00 on card 4711-2233-4455-6677",
        "entities": [
            (0,  16,  "PERSON"),
            (30, 37,  "ORG"),
            (47, 59,  "PHONE"),
            (63, 80,  "EMAIL"),
            (90, 136, "ADDRESS"),
            (147, 154, "AMOUNT"),
            (163, 182, "CREDIT_CARD"),
        ]
    },
    {
        "text": "mike torres 905-334-7821 flagged transaction $15,750.00 "
                "account 00891-003-7712334 sin 527-384-910 "
                "dob 1979-11-03 address 99 wellington st w toronto on m5k 1j3",
        "entities": [
            (0,  11,  "PERSON"),
            (12, 24,  "PHONE"),
            (44, 54,  "AMOUNT"),
            (63, 81,  "ACCOUNT_NO"),
            (86, 97,  "SIN"),
            (102, 112, "DOB"),
            (121, 155, "ADDRESS"),
        ]
    },
    {
        "text": "wire transfer approved for jennifer wu at rbc. "
                "sending $24,500.00 from 00234-002-8812334 "
                "to recipient account 00891-006-4421198 swift royccat2. "
                "her email jen.wu@gmail.com phone 647-221-8834",
        "entities": [
            (28, 39,  "PERSON"),
            (43, 46,  "ORG"),
            (56, 65,  "AMOUNT"),
            (71, 89,  "ACCOUNT_NO"),
            (103, 121, "ACCOUNT_NO"),
            (128, 136, "SWIFT"),
            (148, 165, "EMAIL"),
            (172, 184, "PHONE"),
        ]
    },
    {
        "text": "loan application from roberto sanchez sin 482-331-092 "
                "dob 1990-04-15 employer maple tech inc income $78,000.00 "
                "requested $45,000.00 contact 905-441-2281 roberto.s@gmail.com "
                "address 55 queen st e toronto on m5c 1r6",
        "entities": [
            (20, 35,  "PERSON"),
            (40, 51,  "SIN"),
            (56, 66,  "DOB"),
            (76, 90,  "ORG"),
            (98, 107, "AMOUNT"),
            (118, 127, "AMOUNT"),
            (136, 148, "PHONE"),
            (149, 168, "EMAIL"),
            (177, 210, "ADDRESS"),
        ]
    },
]

# Convert to DocBin
db = DocBin()
for item in REAL_TEST_DOCS:
    doc = nlp.make_doc(item['text'])
    ents = []
    for start, end, label in item['entities']:
        span = doc.char_span(start, end, label=label, 
                             alignment_mode='contract')
        if span:
            ents.append(span)
    doc.ents = ents
    db.add(doc)

db.to_disk('../data/annotated/real_test.spacy')
print(f"✅ Real test set saved — {len(REAL_TEST_DOCS)} handcrafted docs")

✅ Real test set saved — 5 handcrafted docs


## Done : Next: 03_model_training.ipynb